In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import StepLR
import json
from torch.amp.autocast_mode import autocast
import time
from gensim.models import Word2Vec
from tqdm import tqdm
import hashlib
import multiprocessing as mp
from functools import partial
import pickle
from scipy.spatial import KDTree

# Заглушка для GradScaler на CPU
class DummyGradScaler:
    def __init__(self): pass
    def scale(self, loss): return loss
    def step(self, optimizer): optimizer.step()
    def update(self): pass

# Вычисление хэша файлов
def compute_file_hash(file_path):
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

def compute_dataset_hash(files):
    hashes = sorted(compute_file_hash(f) for f in files)
    combined = ''.join(hashes).encode()
    return hashlib.md5(combined).hexdigest()

# Чтение и обработка JSON
def load_block_dictionary(json_path):
    start_time = time.time()
    print("Загрузка словаря блоков...")
    with open(json_path, 'r') as f:
        block_dict = json.load(f)

    token_dict = {}
    for key, entry in tqdm(block_dict.items(), desc="Обработка словаря блоков"):
        key = int(key)
        block_state = entry["name"]
        parts = block_state.replace(']', '').split('[')
        block_name = parts[0]
        states = parts[1].split(',') if len(parts) > 1 else []
        tokens = [block_name] + states
        token_dict[key] = tokens

    missing_keys = set(map(str, block_dict.keys())) - set(map(str, token_dict.keys()))
    if missing_keys:
        print(f"Предупреждение: Отсутствуют ключи в token_dict: {missing_keys}")

    print(f"Словарь блоков загружен за {time.time() - start_time:.2f}s")
    return token_dict, block_dict

# Многопоточная подготовка данных для Word2Vec
def process_structure(structure_file, token_dict):
    structure = np.load(structure_file).astype(np.int64)
    sentence = []
    for x in range(structure.shape[0]):
        for y in range(structure.shape[1]):
            for z in range(structure.shape[2]):
                key = structure[x, y, z].item()
                if key in token_dict:
                    sentence.extend(token_dict[key])
    return sentence if sentence else ['dummy']

def prepare_word2vec_data(structures, token_dict, num_workers=4, cache_file="sentences.pkl"):
    if os.path.exists(cache_file):
        with open(cache_file, 'rb') as f:
            print(f"Загрузка кэшированных предложений из {cache_file}...")
            return pickle.load(f)

    start_time = time.time()
    print("Подготовка данных для Word2Vec...")
    pool = mp.Pool(processes=num_workers)
    process_func = partial(process_structure, token_dict=token_dict)
    sentences = []
    for sentence in tqdm(pool.imap(process_func, structures), total=len(structures), desc="Обработка структур"):
        if sentence and sentence != ['dummy']:
            sentences.append(sentence)
    pool.close()
    pool.join()

    with open(cache_file, 'wb') as f:
        pickle.dump(sentences, f)
    print(f"Подготовка данных завершена за {time.time() - start_time:.2f}s")
    return sentences

def train_word2vec(sentences, token_dict, embedding_dim=16, model_path="word2vec.model", token_dict_path="token_dict.json"):
    start_time = time.time()
    print("Обучение Word2Vec...")
    model = Word2Vec(sentences, vector_size=embedding_dim, window=5, min_count=1, workers=mp.cpu_count())
    model.save(model_path)
    with open(token_dict_path, 'w') as f:
        json.dump({str(k): v for k, v in token_dict.items()}, f)
    print(f"Word2Vec обучен и сохранен за {time.time() - start_time:.2f}s")
    return model

def load_word2vec_model(model_path="word2vec.model", token_dict_path="token_dict.json", dataset_hash_path="dataset_hash.json"):
    if os.path.exists(model_path) and os.path.exists(token_dict_path) and os.path.exists(dataset_hash_path):
        with open(token_dict_path, 'r') as f:
            token_dict = {int(k): v for k, v in json.load(f).items()}
        with open(dataset_hash_path, 'r') as f:
            saved_hash = json.load(f)['hash']
        return saved_hash, Word2Vec.load(model_path), token_dict
    return None, None, None

def structure_to_embeddings(structure, token_dict, w2v_model, embedding_dim=16):
    embedding_structure = np.zeros((structure.shape[0], structure.shape[1], structure.shape[2], embedding_dim))
    for x in range(structure.shape[0]):
        for y in range(structure.shape[1]):
            for z in range(structure.shape[2]):
                key = structure[x, y, z].item()
                if key in token_dict:
                    tokens = token_dict[key]
                    embedding = np.zeros(embedding_dim)
                    count = 0
                    for token in tokens:
                        if token in w2v_model.wv:
                            embedding += w2v_model.wv[token]
                            count += 1
                    if count > 0:
                        embedding_structure[x, y, z] = embedding / count
    return embedding_structure

# Анализ блоков
def analyze_blocks(data_dir, global_blocklist_file, output_file="block_analysis.json"):
    start_time = time.time()
    print("Анализ блоков...")
    block_counts = {}
    total_voxels = 0
    files = []

    for root, _, filenames in os.walk(data_dir):
        for filename in filenames:
            if filename.endswith('_normalized.npy'):
                files.append(os.path.join(root, filename))

    for file in tqdm(files, desc="Анализ файлов"):
        data = np.load(file).astype(np.int64)
        unique, counts = np.unique(data, return_counts=True)
        for block_id, count in zip(unique, counts):
            block_counts[int(block_id)] = block_counts.get(int(block_id), 0) + int(count)
            total_voxels += int(count)

    block_percentages = {block_id: (count / total_voxels) * 100 for block_id, count in block_counts.items()}
    with open(global_blocklist_file, 'r') as f:
        block_dict = json.load(f)
    block_analysis = {
        str(block_id): {
            "name": block_dict.get(str(block_id), {"name": "unknown"}).get("name", "unknown"),
            "count": int(count),
            "percentage": float(block_percentages[block_id]),
            "forbidden": block_dict.get(str(block_id), {"forbidden": False}).get("forbidden", False)
        } for block_id, count in block_counts.items()
    }

    with open(output_file, 'w') as f:
        json.dump(block_analysis, f, indent=4)

    print(f"Анализ блоков завершён за {time.time() - start_time:.2f}s")
    return block_analysis

# Датасет
class MinecraftHouseDataset(Dataset):
    def __init__(self, data_dir, grid_size=32, global_blocklist_file=None, embedding_dim=16, cache_dir="cache"):
        self.data_dir = data_dir
        self.grid_size = grid_size
        self.embedding_dim = embedding_dim
        self.cache_dir = cache_dir
        self.files = []

        start_time = time.time()
        print(f"Сканирование директории {data_dir}...")
        for root, _, filenames in tqdm(os.walk(data_dir), desc="Сканирование"):
            for filename in filenames:
                if filename.endswith('_normalized.npy'):
                    self.files.append(os.path.join(root, filename))
        print(f"Сканирование завершено за {time.time() - start_time:.2f}s, найдено {len(self.files)} файлов")

        if not self.files:
            raise ValueError(f"Не найдено .npy файлов в {data_dir}")

        if global_blocklist_file:
            self.token_dict, self.block_dict = load_block_dictionary(global_blocklist_file)
        else:
            raise ValueError("Требуется global_blocklist_file")

        dataset_hash = compute_dataset_hash(self.files)
        saved_hash, saved_model, saved_token_dict = load_word2vec_model()

        if saved_hash == dataset_hash and saved_model and saved_token_dict:
            print("Загрузка сохранённой модели Word2Vec...")
            self.w2v_model = saved_model
            self.token_dict = saved_token_dict
        else:
            sentences = prepare_word2vec_data(self.files, self.token_dict, num_workers=2, cache_file="sentences.pkl")
            self.w2v_model = train_word2vec(sentences, self.token_dict, embedding_dim)
            with open("token_dict.json", 'w') as f:
                json.dump({str(k): v for k, v in self.token_dict.items()}, f)
            with open("dataset_hash.json", 'w') as f:
                json.dump({"hash": dataset_hash}, f)

        self.block_keys = [int(k) for k in self.block_dict.keys()]
        if os.path.exists("block_embeddings.pt"):
            self.block_embeddings = torch.load("block_embeddings.pt", map_location="cpu")
        else:
            self.block_embeddings = torch.zeros(len(self.block_keys), embedding_dim)
            for i, key in enumerate(self.block_keys):
                tokens = self.token_dict[key]
                embedding = torch.zeros(embedding_dim)
                count = 0
                for token in tokens:
                    if token in self.w2v_model.wv:
                        embedding += torch.tensor(self.w2v_model.wv[token])
                        count += 1
                if count > 0:
                    self.block_embeddings[i] = embedding / count
            torch.save(self.block_embeddings, "block_embeddings.pt")
        self.kdtree = KDTree(self.block_embeddings.numpy())

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        house = np.load(self.files[idx]).astype(np.int64)
        if house.shape != (self.grid_size, self.grid_size, self.grid_size):
            raise ValueError(f"Неверная форма данных: {house.shape}")
        house_emb = structure_to_embeddings(house, self.token_dict, self.w2v_model, self.embedding_dim)
        house_emb = torch.from_numpy(house_emb).float().permute(3, 0, 1, 2)
        return house_emb, torch.from_numpy(house).long()

# VQGAN
class VQGANFullRes(nn.Module):
    def __init__(self, grid_size=32, embedding_dim=16, codebook_size=512):
        super().__init__()
        self.grid_size = grid_size
        self.embedding_dim = embedding_dim
        self.codebook_size = codebook_size
        self.latent_dim = grid_size // 8
        self.encoder = nn.Sequential(
            nn.Conv3d(embedding_dim, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.Conv3d(16, 32, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.Conv3d(32, embedding_dim, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.Conv3d(embedding_dim, embedding_dim, kernel_size=3, stride=1, padding=1),
        )
        self.embedding = nn.Embedding(codebook_size, embedding_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(embedding_dim, embedding_dim, kernel_size=3, stride=1, padding=1), nn.ReLU(),
            nn.ConvTranspose3d(embedding_dim, 32, kernel_size=3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose3d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose3d(16, embedding_dim, kernel_size=3, stride=2, padding=1, output_padding=1),
        )
        self.discriminator = nn.Sequential(
            nn.Conv3d(embedding_dim, 32, kernel_size=4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv3d(32, 64, kernel_size=4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv3d(64, 1, kernel_size=4, stride=1, padding=0), nn.Sigmoid(),
        )

    def quantize(self, z):
        z_flattened = z.permute(0, 2, 3, 4, 1).reshape(-1, self.embedding_dim)
        distances = torch.cdist(z_flattened, self.embedding.weight)
        encoding_indices = torch.argmin(distances, dim=1)
        z_q = self.embedding(encoding_indices).view(z.shape)
        return z_q, encoding_indices

    def forward(self, x):
        z = self.encoder(x)
        z_q, indices = self.quantize(z)
        x_recon = self.decoder(z_q)
        return x_recon, indices

    def discriminate(self, x):
        return self.discriminator(x)

# Discrete Diffusion
class DiscreteDiffusion(nn.Module):
    def __init__(self, codebook_size=512, timesteps=1000, embedding_dim=16, latent_dim=2):
        super().__init__()
        self.codebook_size = codebook_size
        self.timesteps = timesteps
        self.latent_dim = latent_dim
        self.embedding = nn.Embedding(codebook_size, embedding_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embedding_dim, nhead=4, dim_feedforward=512, batch_first=True),
            num_layers=4
        )
        self.output_layer = nn.Linear(embedding_dim, codebook_size)

    def forward(self, indices, t):
        batch_size, seq_len = indices.shape
        mask = torch.rand(batch_size, seq_len, device=indices.device) < (t / self.timesteps).view(-1, 1)
        noisy_indices = torch.where(mask, torch.randint(0, self.codebook_size, indices.shape, device=indices.device), indices)
        x = self.embedding(noisy_indices)
        x = self.transformer(x)
        pred = self.output_layer(x)
        return pred

    def sample(self, batch_size, seq_len, device, block_weights=None):
        if block_weights is None:
            indices = torch.randint(0, self.codebook_size, (batch_size, seq_len), device=device)
        else:
            indices = torch.zeros(batch_size, seq_len, dtype=torch.long, device=device)
            for i in range(batch_size):
                indices[i] = torch.multinomial(block_weights, num_samples=seq_len, replacement=True)
        return indices

# Обучение VQGAN
def train_vqgan_full_res(dataset, num_epochs=5, batch_size=2, accumulation_steps=2):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Используется устройство: {device}")
    vqgan = VQGANFullRes(grid_size=32, embedding_dim=16, codebook_size=512).to(device)
    optimizer_g = torch.optim.Adam(vqgan.parameters(), lr=1e-3)
    optimizer_d = torch.optim.Adam(vqgan.discriminator.parameters(), lr=2e-4)
    scheduler_g = StepLR(optimizer_g, step_size=3, gamma=0.5)
    scheduler_d = StepLR(optimizer_d, step_size=3, gamma=0.5)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else DummyGradScaler()

    g_losses = []
    d_losses = []
    criterion = nn.MSELoss(reduction='none')
    start_time = time.time()
    epoch_times = []

    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        total_loss_g, total_loss_d = 0, 0
        for i, (batch_emb, batch_ids) in enumerate(tqdm(dataloader, desc=f"Эпоха {epoch+1}/{num_epochs}")):
            batch_emb = batch_emb.to(device)
            batch_ids = batch_ids.to(device)
            block_mask = (batch_ids != 0).float()
            optimizer_g.zero_grad()
            optimizer_d.zero_grad()

            with autocast(device_type=device.type):
                recon, _ = vqgan(batch_emb)
                recon_loss = criterion(recon, batch_emb)
                g_loss = 10 * (recon_loss * block_mask.unsqueeze(1)).mean()
                real_out = vqgan.discriminate(batch_emb)
                fake_out = vqgan.discriminate(recon.detach())
                d_loss = -torch.mean(torch.log(real_out + 1e-8) + torch.log(1 - fake_out + 1e-8))

            d_loss = d_loss / accumulation_steps
            scaler.scale(d_loss).backward()
            if (i + 1) % accumulation_steps == 0:
                scaler.step(optimizer_d)
                scaler.update()
                optimizer_d.zero_grad()
                if device.type == 'cuda':
                    torch.cuda.empty_cache()

            g_loss = g_loss / accumulation_steps
            scaler.scale(g_loss).backward()
            if (i + 1) % accumulation_steps == 0:
                scaler.step(optimizer_g)
                scaler.update()
                optimizer_g.zero_grad()
                if device.type == 'cuda':
                    torch.cuda.empty_cache()

            total_loss_g += g_loss.item() * accumulation_steps
            total_loss_d += d_loss.item() * accumulation_steps

        avg_g_loss = total_loss_g / len(dataloader)
        avg_d_loss = total_loss_d / len(dataloader)
        g_losses.append(avg_g_loss)
        d_losses.append(avg_d_loss)

        epoch_time = time.time() - epoch_start_time
        epoch_times.append(epoch_time)
        avg_epoch_time = sum(epoch_times) / len(epoch_times)
        remaining_epochs = num_epochs - (epoch + 1)
        remaining_time = remaining_epochs * avg_epoch_time
        remaining_time_str = time.strftime("%H:%M:%S", time.gmtime(remaining_time))

        print(f"Эпоха {epoch+1}/{num_epochs}, G Loss: {avg_g_loss:.4f}, D Loss: {avg_d_loss:.4f}, Время: {epoch_time:.2f}с, Осталось: {remaining_time_str}")
        scheduler_g.step()
        scheduler_d.step()

    plt.figure(figsize=(10, 5))
    plt.plot(range(1, num_epochs + 1), g_losses, label='Generator Loss')
    plt.plot(range(1, num_epochs + 1), d_losses, label='Discriminator Loss')
    plt.xlabel('Эпоха')
    plt.ylabel('Loss')
    plt.title('Потери при обучении VQGAN')
    plt.legend()
    plt.grid(True)
    plt.savefig('training_loss_vqgan.png')
    plt.close()

    total_time = time.time() - start_time
    print(f"Общее время обучения: {total_time:.2f}с")
    return vqgan

# Обучение Diffusion
def train_diffusion(vqgan_full_res, dataset, num_epochs=5, batch_size=2, accumulation_steps=2):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Используется устройство: {device}")
    diffusion = DiscreteDiffusion(codebook_size=512, timesteps=1000, embedding_dim=16, latent_dim=2).to(device)
    optimizer = torch.optim.Adam(diffusion.parameters(), lr=1e-3)
    scheduler = StepLR(optimizer, step_size=3, gamma=0.5)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else DummyGradScaler()

    losses = []
    start_time = time.time()
    epoch_times = []
    latent_size = vqgan_full_res.latent_dim ** 3

    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        total_loss = 0
        for i, (batch_emb, _) in enumerate(tqdm(dataloader, desc=f"Эпоха {epoch+1}/{num_epochs}")):
            batch_emb = batch_emb.to(device)
            with autocast(device_type=device.type):
                with torch.no_grad():
                    _, indices = vqgan_full_res(batch_emb)
                    indices = indices.view(batch_emb.size(0), latent_size)
                t = torch.randint(0, diffusion.timesteps, (batch_emb.size(0),), device=device)
                pred = diffusion.forward(indices, t)
                loss = F.cross_entropy(pred.view(-1, diffusion.codebook_size), indices.view(-1))
            optimizer.zero_grad()
            loss = loss / accumulation_steps
            scaler.scale(loss).backward()
            if (i + 1) % accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                if device.type == 'cuda':
                    torch.cuda.empty_cache()
            total_loss += loss.item() * accumulation_steps

        avg_loss = total_loss / len(dataloader)
        losses.append(avg_loss)

        epoch_time = time.time() - epoch_start_time
        epoch_times.append(epoch_time)
        avg_epoch_time = sum(epoch_times) / len(epoch_times)
        remaining_epochs = num_epochs - (epoch + 1)
        remaining_time = remaining_epochs * avg_epoch_time
        remaining_time_str = time.strftime("%H:%M:%S", time.gmtime(remaining_time))

        print(f"Эпоха {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}, Время: {epoch_time:.2f}с, Осталось: {remaining_time_str}")
        scheduler.step()

    plt.figure(figsize=(10, 5))
    plt.plot(range(1, num_epochs + 1), losses, label='Diffusion Loss')
    plt.xlabel('Эпоха')
    plt.ylabel('Loss')
    plt.title('Потери при обучении Diffusion')
    plt.legend()
    plt.grid(True)
    plt.savefig('training_loss_diffusion.png')
    plt.close()

    total_time = time.time() - start_time
    print(f"Общее время обучения: {total_time:.2f}с")
    return diffusion

if __name__ == "__main__":
    data_dir = "/content/Data/Output"
    global_blocklist_file = "/content/global_block_list.json"
    block_analysis_file = "/content/block_analysis_shifted.json"
    cache_dir = "cache"
    batch_size = 32
    num_epochs_vqgan = 24
    num_epochs_diffusion = 24
    accumulation_steps = 2
    embedding_dim = 16

    total_start_time = time.time()
    print("Начало выполнения программы...")

    # Анализ блоков
    block_analysis = analyze_blocks(data_dir, global_blocklist_file, block_analysis_file)
    print(f"Анализ блоков сохранён в {block_analysis_file}")

    # Создание датасета
    dataset = MinecraftHouseDataset(
        data_dir,
        grid_size=32,
        global_blocklist_file=global_blocklist_file,
        embedding_dim=embedding_dim,
        cache_dir=cache_dir
    )

    # Обучение VQGAN
    vqgan_full_res = train_vqgan_full_res(dataset, num_epochs=num_epochs_vqgan, batch_size=batch_size, accumulation_steps=accumulation_steps)
    torch.save(vqgan_full_res.state_dict(), "vqgan_full_res_model.pth")

    # Обучение диффузии
    diffusion = train_diffusion(vqgan_full_res, dataset, num_epochs=num_epochs_diffusion, batch_size=batch_size, accumulation_steps=accumulation_steps)
    torch.save(diffusion.state_dict(), "diffusion_model.pth")

    total_time = time.time() - total_start_time
    print(f"Программа завершена за {total_time:.2f}с")